# Sentinel-2 land-cover classification for the East Cameroon mining belt

Reproduces the GEE pipeline of Anaedevha et al. (2026): cloud-masked Sentinel-2 SR composite,
seven manuscript indices, and a 500-tree Random-Forest classifier on 17 features.

**Run requirement:** valid `earthengine authenticate` session and a Google Earth Engine account.

In [ ]:
import ee, geemap
ee.Initialize()
from cmhr.utils import load_config
from cmhr.remote_sensing import build_sentinel2_collection, annual_composite, all_indices, build_gee_random_forest
bbox = load_config('study_area')['bounding_box']
aoi = ee.Geometry.Rectangle([bbox['min_lon'], bbox['min_lat'], bbox['max_lon'], bbox['max_lat']])

In [ ]:
coll = build_sentinel2_collection(aoi, '2024-01-01', '2024-12-31')
comp = annual_composite(coll, 2024)
bands = {'blue':comp.select('B2'),'green':comp.select('B3'),'red':comp.select('B4'),
         'nir':comp.select('B8'),'swir1':comp.select('B11')}
for k, img in all_indices(bands).items():
    comp = comp.addBands(img.rename(k))
comp = comp.select(['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12','NDVI','EVI','SAVI','BSI','NDWI','MNDWI','NDMI'])
comp.bandNames().getInfo()

In [ ]:
# Replace `training_fc` with your labelled feature collection (≥200 points/class).
# from cmhr.remote_sensing import train_lulc_classifier
# classifier = train_lulc_classifier(comp, training_fc, label='class')
# classified = comp.classify(classifier)
# Map = geemap.Map(); Map.centerObject(aoi, 7); Map.addLayer(classified, {'min':0,'max':6}, '2024 LULC'); Map